# What's Inside a Pipeline?

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/Building_with_Deep_Learning/01-llms/03_inside_pipeline.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

_Click the badge above to open and run this notebook in Google Colab!_

In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/Building_with_Deep_Learning/01-llms"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Set up

In [ ]:
!pip install -qqq torch
!pip install -Uqqq transformers datasets evaluate accelerate timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 81.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.5 MB/s eta 0:00:00:00:0100:01


### Suppress output logs

In [ ]:
import os
import logging

from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
disable_progress_bars()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

## Overview

The [`Pipeline`](/docs/transformers/v5.5.0/en/main_classes/pipelines#transformers.Pipeline) is a simple but powerful **inference API** that is readily available for a variety of machine learning **tasks** with any **model** from the Hugging Face [Hub](https://hf.co/models). 

A pipeline wraps three things into one call:

1. **Pre-processing** – Tokenizing raw text (or processing an image/audio input).
2. **Model inference** – Running the fine-tuned model.
3. **Post-processing** – Converting raw logits/outputs into human-readable results (e.g., labels, extracted answers, or generated text).

![](../assets/pipeline.png){.r-stretch fig-align="center"}

You pass in raw data and get back a structured result without touching the underlying tensors.


### Pre- and Post-processing

#### Audio Models

* **Pre-processing:** Resample raw waveforms to the target rate (e.g., 16kHz) and extract/normalize audio features (like log-Mel spectrograms).
* **Post-processing:** Output class probabilities for classification, or decode acoustic states (via beam search/CTC) into text for speech recognition.

#### Text Generative Models

* **Pre-processing:** Tokenize source text into IDs and append necessary task-specific prefixes (e.g., "summarize:").
* **Post-processing:** Generate new tokens autoregressively, decode IDs back to strings, and clean up subword artifacts and special tokens.

#### Image Models

* **Pre-processing:** Resize, crop, and normalize raw images using an `ImageProcessor` to generate tensor arrays (`pixel_values`), and format any task-specific annotations.
* **Post-processing:** filtering confidence scores and scaling bounding boxes for object detection, or decoding pixel-level arrays into semantic maps for segmentation.

## Model Instantiation

The number of user-facing abstractions is limited to only three classes for instantiating a model, and two APIs for inference or training. 

The model instantiation classes are:

1. Preprocessor; a class for converting raw inputs (text, images, audio, multimodal) into numerical inputs to the model. For example, [`PreTrainedTokenizer`](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/tokenizer#transformers.PythonBackend) converts text into tensors and [`ImageProcessingMixin`](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/image_processor#transformers.ImageProcessingMixin) converts pixels into tensors.

2. [`PreTrainedModel`](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/model#transformers.PreTrainedModel) (architecture) with [`PreTrainedConfig`](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/configuration#transformers.PreTrainedConfig).

But the docs recommend using the [`AutoClass`](https://huggingface.co/docs/transformers/model_doc/auto) API to load both components at once.

### Pretrained Model

Use the [`from_pretrained()`](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/model#transformers.PreTrainedModel.from_pretrained) method to also load the **weights** alongside the **configuration** from the **Hub** into the model and preprocessor class.

In [2]:
from transformers import AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-360M"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    
    # `dtype="auto"` directly initializes the model weights in the
    # data type they’re stored in, which can help avoid loading the
    # weights twice (PyTorch loads weights in `torch.float32` by default).
    dtype="auto",

    # `device_map="auto"` automatically allocates the model weights
    # to your fastest device first.
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### Load Tokenizer

A **Tokenizer** is needed to preprocess the input before it is fed to the LLM. Each LLM has it's own specialized tokenizer, so we'll use the same `model_name` to load it with `AutoTokenizer`.

> A **tokenizer** is a deterministic preprocessing component in Natural Language Processing (NLP) responsible for translating raw text into numerical tensors that machine learning models can process. It partitions continuous string sequences into atomic units called _tokens_ (words, subwords, or characters) and maps them to a pre-defined numerical vocabulary.

![](../assets/tokens.png)

![](../assets/token_ids.png)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure the tokenizer has a pad_token.
# If not, assign it to eos_token.
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Tokenize the text and return PyTorch tensors with the tokenizer. Move the model to an accelerator if it’s available to accelerate inference.

In [ ]:
input_text = [
    "the secret lies in the",
    "the man said how",
    "the cat sat on",
    "today we learn about",
]


input_tokens = (
    tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        padding_side='left',
    )
    .to(model.device)
)
input_tokens

{'input_ids': tensor([[ 1195,  4911,  5721,   281,   260],
        [    0,  1195,   555,  1137,   638],
        [    0,  1195,  2644,  2643,   335],
        [    0, 28195,   392,   835,   563]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1],
        [0, 1, 1, 1, 1],
        [0, 1, 1, 1, 1]], device='cuda:0')}

The model is now ready for inference or training.

For inference:

1. Pass tokenized inputs to [generate()](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/text_generation#transformers.GenerationMixin.generate) to generate text.
2. Decode the token ids back into text with [batch_decode()](https://huggingface.co/docs/transformers/v5.12.0/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) (post-processing)

In [11]:
output_tokens = model.generate(**input_tokens, max_length=30)
output_text = tokenizer.batch_decode(output_tokens)
output_text

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


['the secret lies in the fact that the two are not the same.\n\nThe first is the one that is the most obvious.\n\nThe',
 '<|endoftext|>the man said how he had been a soldier in the army of the United States, and had been wounded in the battle of the Alamo,',
 '<|endoftext|>the cat sat on the table and looked at me.\n\n"I\'m sorry, I didn\'t mean to startle you."\n\n',
 '<|endoftext|>today we learn about the importance of the 10 commandments.\n\nThe 10 commandments are the first 10 words of the']